# Question 1: Tackling Mode Collapse in GANs
## DCGAN vs WGAN-GP 

## 1. Install Dependencies

In [1]:
import os
import torch

def save_checkpoint(state, filename):
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    torch.save(state, filename)

def load_checkpoint(path, netG, netD, optimG, optimD, scaler=None):
    checkpoint = torch.load(path)
    netG.load_state_dict(checkpoint['netG'])
    netD.load_state_dict(checkpoint['netD'])
    optimG.load_state_dict(checkpoint['optimG'])
    optimD.load_state_dict(checkpoint['optimD'])
    if scaler and 'scaler' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler'])
    start_epoch = checkpoint['epoch']
    start_iter = checkpoint.get('iter', 0)
    print(f"Resumed from Epoch {start_epoch}, Iter {start_iter}")
    return start_epoch, start_iter


In [2]:
# ===== Resume Training Config =====
start_epoch = 0
start_iter = 0
RESUME_PATH = None  # e.g., 'checkpoints/dcgan_latest.pth'

if RESUME_PATH:
    start_epoch, start_iter = load_checkpoint(
        RESUME_PATH, netG_dc, netD_dc, optimG_dc, optimD_dc, scaler
    )


In [3]:
!pip install -q gradio torch torchvision matplotlib numpy Pillow scikit-image

## 2. Imports & Configuration

In [4]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device Setup (Multi-GPU) ─────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f'Device : {device}')
print(f'GPUs   : {num_gpus}')

# ── Hyperparameters ──────────────────────────────────────────
IMG_SIZE    = 64
NZ          = 100          # noise vector dim
NGF         = 64           # generator feature maps
NDF         = 64           # discriminator feature maps
NC          = 3            # colour channels
BATCH_SIZE  = 64
LR          = 0.0002
BETAS       = (0.5, 0.999)
EPOCHS_DCGAN   = 50
EPOCHS_WGAN    = 50
LAMBDA_GP   = 10           # gradient penalty weight
N_CRITIC    = 5            # critic updates per generator update
SUBSET_SIZE = 8000         # use subset for speed

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('generated',   exist_ok=True)
print('Config ready.')

Device : cuda
GPUs   : 2
Config ready.


## 3. Data Preparation

In [5]:
import os, random
from PIL import Image

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import kagglehub

BATCH_SIZE = 64
SUBSET_SIZE = None

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ── Download datasets ─────────────────────────────
pokemon_path = kagglehub.dataset_download("jackemartin/pokemon-sprites")
anime_path   = kagglehub.dataset_download("soumikrakshit/anime-faces")

print("Pokemon path:", pokemon_path)
print("Anime path:", anime_path)

# ── Auto-detect actual image folders ─────────────
def find_image_root(base_path):
    for root, _, files in os.walk(base_path):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                return root
    return base_path

ANIME_DIR = find_image_root(anime_path)
POKEMON_DIR = find_image_root(pokemon_path)

print("Resolved Anime dir:", ANIME_DIR)
print("Resolved Pokemon dir:", POKEMON_DIR)

# ── Loader function ──────────────────────────────
def build_loader(root, subset=None):
    if not os.path.exists(root):
        raise RuntimeError(f"Dataset path does not exist: {root}")

    ds = None
    try:
        temp_ds = datasets.ImageFolder(root, transform=transform)
        if len(temp_ds) > 0:
            ds = temp_ds
    except Exception:
        pass

    if ds is None or len(ds) == 0:
        from torch.utils.data import Dataset

        class FlatDataset(Dataset):
            def __init__(self, folder, tfm):
                exts = ('.png', '.jpg', '.jpeg', '.webp')
                self.imgs = [
                    os.path.join(dp, f)
                    for dp, _, files in os.walk(folder)
                    for f in files if f.lower().endswith(exts)
                ]
                self.tfm = tfm

            def __len__(self):
                return len(self.imgs)

            def __getitem__(self, i):
                img = Image.open(self.imgs[i]).convert('RGB')
                return self.tfm(img), 0

        ds = FlatDataset(root, transform)

    if len(ds) == 0:
        raise RuntimeError(f"No images found in '{root}'")

    if subset and len(ds) > subset:
        idx = random.sample(range(len(ds)), subset)
        ds = Subset(ds, idx)

    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )

    print(f"Loaded {len(ds)} images from: {root}")
    return loader

# ── Load datasets ────────────────────────────────
anime_loader = None
pokemon_loader = None

try:
    anime_loader = build_loader(ANIME_DIR, SUBSET_SIZE)
except Exception as e:
    print(f"[WARNING] Anime dataset: {e}")

try:
    pokemon_loader = build_loader(POKEMON_DIR, SUBSET_SIZE)
except Exception as e:
    print(f"[WARNING] Pokemon dataset: {e}")

if anime_loader is None and pokemon_loader is None:
    raise ValueError("Neither dataset loaded. Check dataset structure.")

primary_loader = anime_loader if anime_loader is not None else pokemon_loader
dataset_name = "Anime" if anime_loader is not None else "Pokemon"

print(f"Using {dataset_name} dataset")
print(f"Total images: {len(primary_loader.dataset)}")
print(f"Batches per epoch: {len(primary_loader)}")

Pokemon path: /kaggle/input/datasets/jackemartin/pokemon-sprites
Anime path: /kaggle/input/datasets/soumikrakshit/anime-faces
Resolved Anime dir: /kaggle/input/datasets/soumikrakshit/anime-faces/data
Resolved Pokemon dir: /kaggle/input/datasets/jackemartin/pokemon-sprites/pokemon_images/pokemondb.net
Loaded 21551 images from: /kaggle/input/datasets/soumikrakshit/anime-faces/data
Loaded 33887 images from: /kaggle/input/datasets/jackemartin/pokemon-sprites/pokemon_images/pokemondb.net
Using Anime dataset
Total images: 21551
Batches per epoch: 336


## 4. Model Architectures

In [6]:
# ─────────────────────────────────────────────────────────────
#  Weight initialisation (DCGAN paper)
# ─────────────────────────────────────────────────────────────
def weights_init(m):
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# ─────────────────────────────────────────────────────────────
#  DCGAN Generator
# ─────────────────────────────────────────────────────────────
class DCGenerator(nn.Module):
    def __init__(self, nz=NZ, ngf=NGF, nc=NC):
        super().__init__()
        self.main = nn.Sequential(
            # z → 4×4
            nn.ConvTranspose2d(nz,  ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            # 4 → 8
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            # 8 → 16
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            # 16 → 32
            nn.ConvTranspose2d(ngf*2, ngf,   4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
            # 32 → 64
            nn.ConvTranspose2d(ngf,  nc,     4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, x): return self.main(x)

# ─────────────────────────────────────────────────────────────
#  DCGAN Discriminator
# ─────────────────────────────────────────────────────────────
class DCDiscriminator(nn.Module):
    def __init__(self, nc=NC, ndf=NDF):
        super().__init__()
        self.main = nn.Sequential(
            # 64 → 32
            nn.Conv2d(nc,    ndf,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),
            # 32 → 16
            nn.Conv2d(ndf,   ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, True),
            # 16 → 8
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, True),
            # 8 → 4
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8), nn.LeakyReLU(0.2, True),
            # 4 → 1
            nn.Conv2d(ndf*8, 1,     4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x): return self.main(x)

# ─────────────────────────────────────────────────────────────
#  WGAN-GP Generator  (same arch, different training)
# ─────────────────────────────────────────────────────────────
class WGenerator(nn.Module):
    def __init__(self, nz=NZ, ngf=NGF, nc=NC):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz,  ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf,   4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),   nn.ReLU(True),
            nn.ConvTranspose2d(ngf,  nc,     4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, x): return self.main(x)

# ─────────────────────────────────────────────────────────────
#  WGAN-GP Critic  (no Sigmoid — outputs raw scores)
# ─────────────────────────────────────────────────────────────
class WCritic(nn.Module):
    def __init__(self, nc=NC, ndf=NDF):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(nc,    ndf,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf,   ndf*2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(ndf*8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf*8, 1,     4, 1, 0, bias=False)
            # NO Sigmoid
        )
    def forward(self, x): return self.main(x)

print('Model classes defined.')

Model classes defined.


## 5. Helper Utilities

In [7]:
def make_noise(n, nz=NZ):
    return torch.randn(n, nz, 1, 1, device=device)

fixed_noise = make_noise(16)

def show_generated(netG, noise=None, title='Generated', n=16, nz=NZ, denorm=True):
    netG.eval()
    with torch.no_grad():
        z    = noise if noise is not None else make_noise(n, nz)
        imgs = netG(z[:n]).cpu()
    if denorm:
        imgs = imgs * 0.5 + 0.5
    grid = make_grid(imgs, nrow=4, padding=2)
    plt.figure(figsize=(8, 8))
    plt.title(title, fontsize=14)
    plt.imshow(grid.permute(1,2,0).clamp(0,1).numpy())
    plt.axis('off'); plt.tight_layout(); plt.show()
    netG.train()

def save_checkpoint(state, path):
    torch.save(state, path)

def plot_losses(g_losses, d_losses, title='Training Loss'):
    plt.figure(figsize=(10, 4))
    plt.plot(g_losses, label='Generator Loss',     color='blue')
    plt.plot(d_losses, label='Discriminator Loss', color='red')
    plt.xlabel('Iteration'); plt.ylabel('Loss')
    plt.title(title); plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()

# Gradient Penalty for WGAN-GP
def gradient_penalty(critic, real, fake, device):
    B, C, H, W = real.shape
    alpha = torch.rand(B, 1, 1, 1, device=device).expand_as(real)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = critic(interp)
    grads = autograd.grad(
        outputs=d_interp, inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    grads = grads.view(B, -1)
    gp    = ((grads.norm(2, dim=1) - 1) ** 2).mean()
    return gp

print('Utilities ready.')

Utilities ready.


## 6. DCGAN Training

In [8]:
# ======== INITIALIZATION (MUST BE BEFORE RESUME) ========

netG_dc = DCGenerator().to(device)
netD_dc = DCDiscriminator().to(device)

if num_gpus > 1:
    netG_dc = nn.DataParallel(netG_dc)
    netD_dc = nn.DataParallel(netD_dc)

netG_dc.apply(weights_init)
netD_dc.apply(weights_init)

criterion = nn.BCEWithLogitsLoss()

optimG_dc = optim.Adam(netG_dc.parameters(), lr=LR, betas=BETAS)
optimD_dc = optim.Adam(netD_dc.parameters(), lr=LR, betas=BETAS)

scaler = GradScaler()


# ======== RESUME (NOW SAFE) ========

start_epoch = 0
start_iter = 0

RESUME_PATH = "checkpoints/dcgan_latest.pth"  # or None

if RESUME_PATH and os.path.exists(RESUME_PATH):
    start_epoch, start_iter = load_checkpoint(
        RESUME_PATH, netG_dc, netD_dc, optimG_dc, optimD_dc, scaler
    )


print('Starting DCGAN Training...')


# ======== TRAINING LOOP ========

for epoch in range(start_epoch, EPOCHS_DCGAN):

    epoch_g, epoch_d = 0.0, 0.0

    for i, (real_imgs, _) in enumerate(primary_loader):

        if epoch == start_epoch and i < start_iter:
            continue

        B = real_imgs.size(0)
        real_imgs = real_imgs.to(device)

        # ======== Train Discriminator ========
        optimD_dc.zero_grad()

        label_real = torch.full((B,), 0.9, device=device)
        label_fake = torch.full((B,), 0.0, device=device)

        with autocast():
            out_real = netD_dc(real_imgs).view(-1)
            lossD_real = criterion(out_real, label_real)

            noise = make_noise(B)
            fake = netG_dc(noise)

            out_fake = netD_dc(fake.detach()).view(-1)
            lossD_fake = criterion(out_fake, label_fake)

            lossD = lossD_real + lossD_fake

        scaler.scale(lossD).backward()
        scaler.step(optimD_dc)

        # ======== Train Generator ========
        optimG_dc.zero_grad()

        label_gen = torch.full((B,), 0.9, device=device)

        with autocast():
            out_gen = netD_dc(fake).view(-1)
            lossG = criterion(out_gen, label_gen)

        scaler.scale(lossG).backward()
        scaler.step(optimG_dc)
        scaler.update()

        epoch_g += lossG.item()
        epoch_d += lossD.item()

        # 🔥 SAVE EVERY 200 STEPS
        if i % 200 == 0:
            save_checkpoint({
                'epoch': epoch,
                'iter': i,
                'netG': netG_dc.state_dict(),
                'netD': netD_dc.state_dict(),
                'optimG': optimG_dc.state_dict(),
                'optimD': optimD_dc.state_dict(),
                'scaler': scaler.state_dict()
            }, 'checkpoints/dcgan_latest.pth')

    avg_g = epoch_g / len(primary_loader)
    avg_d = epoch_d / len(primary_loader)

    print(f'[DCGAN] Epoch [{epoch+1}/{EPOCHS_DCGAN}] Loss_G: {avg_g:.4f} Loss_D: {avg_d:.4f}')

    # 🔥 SAVE EVERY EPOCH
    save_checkpoint({
        'epoch': epoch + 1,
        'iter': 0,
        'netG': netG_dc.state_dict(),
        'netD': netD_dc.state_dict(),
        'optimG': optimG_dc.state_dict(),
        'optimD': optimD_dc.state_dict(),
        'scaler': scaler.state_dict()
    }, f'checkpoints/dcgan_epoch_{epoch+1}.pth')

    torch.cuda.empty_cache()

print('DCGAN Training Complete!')

Starting DCGAN Training...
[DCGAN] Epoch [1/50] Loss_G: 0.6931 Loss_D: 1.1102
[DCGAN] Epoch [2/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [3/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [4/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [5/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [6/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [7/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [8/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [9/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [10/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [11/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [12/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [13/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [14/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [15/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [16/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [17/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [18/50] Loss_G: 0.6931 Loss_D: 1.1064
[DCGAN] Epoch [19/50] Loss_G: 0.6931 Loss_D: 1

## 7. DCGAN — Training Logs & Samples

In [9]:
plot_losses(g_losses_dc, d_losses_dc, 'DCGAN — Generator vs Discriminator Loss')
show_generated(netG_dc, fixed_noise, 'DCGAN — Final Generated Images (16 samples)', n=16)

NameError: name 'g_losses_dc' is not defined

## 8. WGAN-GP Training

In [ ]:
# ======== INITIALIZATION ========

netG = DCGenerator().to(device)
netD = DCDiscriminator().to(device)  # acts as Critic (NO sigmoid)

if num_gpus > 1:
    netG = nn.DataParallel(netG)
    netD = nn.DataParallel(netD)

netG.apply(weights_init)
netD.apply(weights_init)

optimG = optim.Adam(netG.parameters(), lr=LR, betas=BETAS)
optimD = optim.Adam(netD.parameters(), lr=LR, betas=BETAS)

# ======== HYPERPARAMETERS ========
lambda_gp = 10
CRITIC_ITER = 3   # reduced for stability


# ======== GRADIENT PENALTY ========
def gradient_penalty(critic, real, fake):
    B, C, H, W = real.shape
    epsilon = torch.rand(B, 1, 1, 1).to(device)
    epsilon = epsilon.expand_as(real)

    interpolated = real * epsilon + fake * (1 - epsilon)
    interpolated.requires_grad_(True)

    mixed_scores = critic(interpolated)

    grad = torch.autograd.grad(
        inputs=interpolated,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True
    )[0]

    grad = grad.view(grad.shape[0], -1)
    grad_norm = grad.norm(2, dim=1)

    gp = torch.mean((grad_norm - 1) ** 2)
    return gp


# ======== RESUME SUPPORT ========

start_epoch = 0
start_iter = 0

RESUME_PATH = "checkpoints/wgan_latest.pth"

if RESUME_PATH and os.path.exists(RESUME_PATH):
    start_epoch, start_iter = load_checkpoint(
        RESUME_PATH, netG, netD, optimG, optimD
    )

print("Starting WGAN-GP Training...")


# ======== TRAINING LOOP ========

for epoch in range(start_epoch, EPOCHS_WGAN):

    epoch_g, epoch_d = 0.0, 0.0

    for i, (real_imgs, _) in enumerate(primary_loader):

        if epoch == start_epoch and i < start_iter:
            continue

        real_imgs = real_imgs.to(device)
        B = real_imgs.size(0)

        # ======== TRAIN CRITIC ========
        for _ in range(CRITIC_ITER):

            noise = make_noise(B)
            fake = netG(noise)

            critic_real = netD(real_imgs).view(-1)
            critic_fake = netD(fake.detach()).view(-1)

            gp = gradient_penalty(netD, real_imgs, fake)

            lossD = -(torch.mean(critic_real) - torch.mean(critic_fake)) + lambda_gp * gp

            optimD.zero_grad()
            lossD.backward()
            optimD.step()

        # ======== TRAIN GENERATOR ========
        noise = make_noise(B)
        fake = netG(noise)
        gen_fake = netD(fake).view(-1)

        lossG = -torch.mean(gen_fake)

        optimG.zero_grad()
        lossG.backward()
        optimG.step()

        epoch_g += lossG.item()
        epoch_d += lossD.item()

        # 🔥 SAVE EVERY 100 STEPS (IMPORTANT)
        if i % 100 == 0:
            save_checkpoint({
                'epoch': epoch,
                'iter': i,
                'netG': netG.state_dict(),
                'netD': netD.state_dict(),
                'optimG': optimG.state_dict(),
                'optimD': optimD.state_dict()
            }, 'checkpoints/wgan_latest.pth')

    avg_g = epoch_g / len(primary_loader)
    avg_d = epoch_d / len(primary_loader)

    print(f'[WGAN-GP] Epoch [{epoch+1}/{EPOCHS_WGAN}] Loss_G: {avg_g:.4f} Loss_D: {avg_d:.4f}')

    # 🔥 SAVE EVERY EPOCH
    save_checkpoint({
        'epoch': epoch + 1,
        'iter': 0,
        'netG': netG.state_dict(),
        'netD': netD.state_dict(),
        'optimG': optimG.state_dict(),
        'optimD': optimD.state_dict()
    }, f'checkpoints/wgan_epoch_{epoch+1}.pth')

    torch.cuda.empty_cache()

print("WGAN-GP Training Complete!")

## 9. WGAN-GP — Training Logs & Samples

In [ ]:
plot_losses(g_losses_wgan, c_losses_wgan, 'WGAN-GP — Generator vs Critic Loss')
show_generated(netG_wgan, fixed_noise, 'WGAN-GP — Final Generated Images (16 samples)', n=16)

## 10. Side-by-Side Comparison (DCGAN vs WGAN-GP)

In [ ]:
def compare_models(netG1, netG2, noise, label1='DCGAN', label2='WGAN-GP', n=8):
    for net in [netG1, netG2]: net.eval()
    with torch.no_grad():
        imgs1 = (netG1(noise[:n]).cpu() * 0.5 + 0.5).clamp(0,1)
        imgs2 = (netG2(noise[:n]).cpu() * 0.5 + 0.5).clamp(0,1)

    fig, axes = plt.subplots(2, n, figsize=(2*n, 5))
    fig.suptitle('DCGAN  (top)  vs  WGAN-GP  (bottom)', fontsize=14)
    for j in range(n):
        axes[0,j].imshow(imgs1[j].permute(1,2,0)); axes[0,j].axis('off')
        axes[1,j].imshow(imgs2[j].permute(1,2,0)); axes[1,j].axis('off')
    axes[0,0].set_ylabel(label1, fontsize=11)
    axes[1,0].set_ylabel(label2, fontsize=11)
    plt.tight_layout(); plt.show()

compare_models(netG_dc, netG_wgan, fixed_noise, n=8)

# Loss curves side by side
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
axs[0].plot(g_losses_dc, label='G Loss', color='blue')
axs[0].plot(d_losses_dc, label='D Loss', color='red')
axs[0].set_title('DCGAN Loss Curves'); axs[0].legend(); axs[0].grid(True)
axs[1].plot(g_losses_wgan, label='G Loss', color='blue')
axs[1].plot(c_losses_wgan, label='C Loss', color='red')
axs[1].set_title('WGAN-GP Loss Curves'); axs[1].legend(); axs[1].grid(True)
plt.suptitle('Training Loss Comparison', fontsize=13)
plt.tight_layout(); plt.show()

## 11. Quantitative Evaluation (Diversity + Optional FID/IS)

In [ ]:
from skimage.metrics import structural_similarity as ssim_fn

def pixel_diversity(netG, n=100):
    """Mean pairwise pixel std across generated samples — proxy for diversity."""
    netG.eval()
    with torch.no_grad():
        z    = make_noise(n)
        imgs = (netG(z).cpu() * 0.5 + 0.5).clamp(0,1).numpy()
    # imgs: (n, 3, 64, 64) → pixel std across batch per channel
    return imgs.std(axis=0).mean()

div_dc   = pixel_diversity(netG_dc)
div_wgan = pixel_diversity(netG_wgan)

print('='*50)
print(f'  Pixel Diversity (higher = more diverse)')
print(f'  DCGAN   : {div_dc:.4f}')
print(f'  WGAN-GP : {div_wgan:.4f}')
print('='*50)

# Optional: FID (requires clean-fid or torch-fidelity)
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    print('\nFID metric available — computing...')
    fid = FrechetInceptionDistance(feature=2048).to(device)
    # Real images
    real_batch, _ = next(iter(anime_loader))
    real_batch = ((real_batch * 0.5 + 0.5) * 255).byte().to(device)
    fid.update(real_batch, real=True)
    # DCGAN fake
    with torch.no_grad():
        fake_dc = netG_dc(make_noise(BATCH_SIZE))
    fake_dc = ((fake_dc * 0.5 + 0.5) * 255).byte()
    fid.update(fake_dc.to(device), real=False)
    print(f'  FID (DCGAN)   : {fid.compute():.2f}')
    fid.reset()
    fid.update(real_batch, real=True)
    with torch.no_grad():
        fake_wg = netG_wgan(make_noise(BATCH_SIZE))
    fake_wg = ((fake_wg * 0.5 + 0.5) * 255).byte()
    fid.update(fake_wg.to(device), real=False)
    print(f'  FID (WGAN-GP) : {fid.compute():.2f}')
except Exception as e:
    print(f'FID skipped ({e})')

## 12. Gradio App Deployment

In [ ]:
import gradio as gr
from PIL import Image as PILImage

# Move models to eval
netG_dc.eval();  netG_wgan.eval()

def generate_images(model_choice: str, num_images: int, seed: int):
    torch.manual_seed(seed)
    n   = int(num_images)
    z   = make_noise(n)
    net = netG_dc if model_choice == 'DCGAN' else netG_wgan
    with torch.no_grad():
        imgs = (net(z).cpu() * 0.5 + 0.5).clamp(0,1)
    grid = make_grid(imgs, nrow=min(n, 4), padding=2)
    pil  = PILImage.fromarray((grid.permute(1,2,0).numpy()*255).astype(np.uint8))
    return pil

def compare_both(num_images: int, seed: int):
    torch.manual_seed(seed)
    n  = int(num_images)
    z  = make_noise(n)
    results = []
    for net, name in [(netG_dc, 'DCGAN'), (netG_wgan, 'WGAN-GP')]:
        with torch.no_grad():
            imgs = (net(z).cpu() * 0.5 + 0.5).clamp(0,1)
        grid = make_grid(imgs, nrow=min(n,4), padding=2)
        pil  = PILImage.fromarray((grid.permute(1,2,0).numpy()*255).astype(np.uint8))
        results.append(pil)
    return results[0], results[1]

with gr.Blocks(title='GAN Image Generator — AI4009') as demo:
    gr.Markdown('# 🎨 GAN Image Generator\n### DCGAN vs WGAN-GP | AI4009 Assignment 3')

    with gr.Tab('Single Model'):
        model_dd  = gr.Dropdown(['DCGAN','WGAN-GP'], value='DCGAN', label='Model')
        n_slider  = gr.Slider(1, 16, value=8, step=1, label='Number of Images')
        seed_sl   = gr.Slider(0, 9999, value=42, step=1, label='Random Seed')
        gen_btn   = gr.Button('Generate', variant='primary')
        out_img   = gr.Image(label='Generated Images')
        gen_btn.click(generate_images, [model_dd, n_slider, seed_sl], out_img)

    with gr.Tab('Compare Both Models'):
        n_slider2  = gr.Slider(1, 8, value=4, step=1, label='Number of Images')
        seed_sl2   = gr.Slider(0, 9999, value=42, step=1, label='Random Seed')
        cmp_btn    = gr.Button('Compare', variant='primary')
        with gr.Row():
            dc_out = gr.Image(label='DCGAN Output')
            wg_out = gr.Image(label='WGAN-GP Output')
        cmp_btn.click(compare_both, [n_slider2, seed_sl2], [dc_out, wg_out])

    with gr.Tab('Model Info'):
        gr.Markdown("""
        ## Model Details
        | Feature | DCGAN | WGAN-GP |
        |---|---|---|
        | Loss | Binary Cross-Entropy | Wasserstein + Gradient Penalty |
        | Discriminator Output | Sigmoid (0–1) | Raw score (no activation) |
        | Training Stability | Moderate | High |
        | Mode Collapse Risk | Higher | Lower |
        | Gradient Penalty λ | — | 10 |
        | Critic Updates / G Update | 1 | 5 |
        """)

demo.launch(share=True)

In [ ]:
# ===== Memory Safety =====
import torch
def free_memory():
    torch.cuda.empty_cache()


In [ ]:
# ===== Anti-Blur DCGAN Tweaks =====
LR_G = 0.0001
LR_D = 0.0004
real_label = 0.9
fake_label = 0.0
print("Applied anti-blur settings")


In [ ]:
# ===== Stable WGAN-GP Training Loop =====
lambda_gp = 10
CRITIC_ITER = 3

def gradient_penalty(critic, real, fake, device):
    B, C, H, W = real.shape
    epsilon = torch.rand(B, 1, 1, 1).repeat(1, C, H, W).to(device)
    interpolated = real * epsilon + fake * (1 - epsilon)
    interpolated.requires_grad_(True)

    mixed_scores = critic(interpolated)

    grad = torch.autograd.grad(
        inputs=interpolated,
        outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True,
        retain_graph=True
    )[0]

    grad = grad.view(grad.shape[0], -1)
    grad_norm = grad.norm(2, dim=1)
    gp = torch.mean((grad_norm - 1) ** 2)
    return gp

print("WGAN-GP setup ready (no AMP, stable config)")


In [ ]:
# ===== Loss Plotting =====
import matplotlib.pyplot as plt

def plot_losses(g_losses, d_losses, title):
    plt.figure()
    plt.plot(g_losses, label="Generator")
    plt.plot(d_losses, label="Discriminator/Critic")
    plt.legend()
    plt.title(title)
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.show()

print("Plot function ready")
